# Launch sites on an interactive map

Proximity coordinates come from OpenStreetMap, looked up for CCAFS SLC-40.

In [1]:
from math import atan2, cos, radians, sin, sqrt

import folium
import pandas as pd
from folium.features import DivIcon
from folium.plugins import MarkerCluster

df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")[["Launch Site", "Lat", "Long", "class"]]
sites = df.groupby("Launch Site", as_index=False).agg(Lat=("Lat", "first"), Long=("Long", "first"), Launches=("class", "size"), Successes=("class", "sum"))
sites

,Launch Site,Lat,Long,Launches,Successes
0,CCAFS LC-40,28.562302,-80.577356,26,7
1,CCAFS SLC-40,28.563197,-80.576820,7,3
2,KSC LC-39A,28.573255,-80.646895,13,10
3,VAFB SLC-4E,34.632834,-120.610745,10,4


In [2]:
def label(point, name, color="#0B3C49"):
    return folium.Marker(point, icon=DivIcon(icon_size=(170, 20), icon_anchor=(0, 0),
                         html=f'<div style="font-size:13px;font-weight:bold;color:{color};white-space:nowrap">{name}</div>'))


sites_map = folium.Map(location=[30, -97], zoom_start=4, tiles="OpenStreetMap")
for _, s in sites.iterrows():
    folium.Circle([s["Lat"], s["Long"]], radius=30000, color="#C8452F", fill=True).add_child(folium.Popup(s["Launch Site"])).add_to(sites_map)
for name, dy in [("CCAFS LC-40", 0.9), ("CCAFS SLC-40", -0.3), ("KSC LC-39A", 2.1), ("VAFB SLC-4E", 0.9)]:
    s = sites[sites["Launch Site"] == name].iloc[0]
    label([s["Lat"] + dy, s["Long"] + 0.7], name, "#C8452F").add_to(sites_map)
sites_map.save("map_sites.html")
sites_map

In [3]:
outcome_map = folium.Map(location=[28.59, -80.60], zoom_start=12, tiles="OpenStreetMap")
cluster = MarkerCluster().add_to(outcome_map)
for _, r in df.iterrows():
    color = "green" if r["class"] == 1 else "red"
    folium.Marker([r["Lat"], r["Long"]], icon=folium.Icon(color=color, icon="rocket", prefix="fa")).add_to(cluster)
outcome_map.save("map_outcomes.html")
outcome_map

In [4]:
def distance(a, b, c, d):
    a, b, c, d = map(radians, (a, b, c, d))
    h = sin((c - a) / 2) ** 2 + cos(a) * cos(c) * sin((d - b) / 2) ** 2
    return 6373.0 * 2 * atan2(sqrt(h), sqrt(1 - h))


site = [28.563197, -80.576820]
places = {"Coastline": [28.5654777, -80.5683199], "Rail line": [28.5634058, -80.5767773],
          "Samuel C Phillips Pkwy": [28.5628619, -80.5707158], "City of Cape Canaveral": [28.3877355, -80.6022846]}
rows = [{"Feature": k, "Distance (km)": round(distance(*site, *v), 2)} for k, v in places.items()]
pd.DataFrame(rows)

,Feature,Distance (km)
0,Coastline,0.87
1,Rail line,0.02
2,Samuel C Phillips Pkwy,0.60
3,City of Cape Canaveral,19.67


In [5]:
near_map = folium.Map(location=[28.5, -80.58], zoom_start=11, tiles="OpenStreetMap")
folium.Circle(site, radius=250, color="#C8452F", fill=True).add_to(near_map)
label(site, "  CCAFS SLC-40", "#C8452F").add_to(near_map)
for r, (name, point) in zip(rows, places.items()):
    folium.PolyLine([site, point], color="#0B3C49", weight=2).add_to(near_map)
    folium.CircleMarker(point, radius=4, color="#0B3C49", fill=True).add_to(near_map)
offsets = {"Coastline": [0.012, 0.012], "Rail line": [0.035, -0.12], "Samuel C Phillips Pkwy": [-0.03, 0.012], "City of Cape Canaveral": [0, 0.01]}
for r, (name, point) in zip(rows, places.items()):
    p = [point[0] + offsets[name][0], point[1] + offsets[name][1]]
    label(p, f"{name}: {r['Distance (km)']} km").add_to(near_map)
near_map.save("map_proximity.html")
near_map.location = site
near_map.options["zoom"] = 15
near_map.save("map_proximity_zoom.html")
near_map